# Phase 2: SQL — Querying the Experiment Data

## Load data into SQLite

In [1]:
import pandas as pd
import sqlite3

# Connect to (or create) a local database file
conn = sqlite3.connect('search_experiment.db')

# Load our CSV
df = pd.read_csv('search_experiment.csv')

# Write it into a SQL table called 'search_events'
df.to_sql('search_events', conn, if_exists='replace', index=False)

print("Table created successfully!")
print(f"Total rows in database: {pd.read_sql('SELECT COUNT(*) as total FROM search_events', conn).iloc[0,0]}")

Table created successfully!
Total rows in database: 20013


In [4]:
print(df.columns.tolist())

['user_id', 'group', 'day', 'clicked', 'dwell_time_sec', 'bounced', 'click_position']


## Query 1: Overall CTR per group

In [3]:
query1 = """
SELECT 
    "group",
    COUNT(*) as total_sessions,
    SUM(clicked) as total_clicks,
    ROUND(AVG(clicked) * 100, 2) as ctr_percent
FROM search_events
GROUP BY "group"
"""

ctr_summary = pd.read_sql(query1, conn)
print("=== Click Through Rate by Group ===")
print(ctr_summary)
ctr_summary.to_csv('result_ctr.csv', index=False)

=== Click Through Rate by Group ===
       group  total_sessions  total_clicks  ctr_percent
0    control           10082          4888        48.48
1  treatment            9931          5451        54.89


The new ranking algorithm is getting more users to click. But right now we don't know if this is real or just luck — that's what Phase 3 (statistics) will tell us.

##  Query 2: Average dwell time per group (only clicked sessions)

In [5]:
query2 = """
SELECT 
    "group",
    ROUND(AVG(dwell_time_sec), 2) as avg_dwell_time,
    ROUND(MIN(dwell_time_sec), 2) as min_dwell,
    ROUND(MAX(dwell_time_sec), 2) as max_dwell
FROM search_events
WHERE clicked = 1
GROUP BY "group"
"""

dwell_summary = pd.read_sql(query2, conn)
print("=== Dwell Time by Group (clicked sessions only) ===")
print(dwell_summary)
dwell_summary.to_csv('result_dwell.csv', index=False)

=== Dwell Time by Group (clicked sessions only) ===
       group  avg_dwell_time  min_dwell  max_dwell
0    control          151.29        5.0      362.2
1  treatment          180.71        5.0      385.7


## Query 3: Bounce rate per group

In [7]:
query3 = """
SELECT 
    "group",
    ROUND(AVG(bounced) * 100, 2) as bounce_rate_percent
FROM search_events
GROUP BY "group"
"""

bounce_summary = pd.read_sql(query3, conn)
print("=== Bounce Rate by Group ===")
print(bounce_summary)
bounce_summary.to_csv('result_bounce.csv', index=False)

=== Bounce Rate by Group ===
       group  bounce_rate_percent
0    control                34.88
1  treatment                25.17


##  Query 4: CTR trend by day (did behavior change over 21 days?)

In [8]:
query4 = """
SELECT 
    day,
    "group",
    ROUND(AVG(clicked) * 100, 2) as daily_ctr
FROM search_events
GROUP BY day, "group"
ORDER BY day
"""

daily_trend = pd.read_sql(query4, conn)
print("=== Daily CTR Trend ===")
print(daily_trend.head(10))
daily_trend.to_csv('result_daily_trend.csv', index=False)

=== Daily CTR Trend ===
   day      group  daily_ctr
0    1    control      47.31
1    1  treatment      54.19
2    2    control      48.76
3    2  treatment      53.39
4    3    control      51.72
5    3  treatment      55.56
6    4    control      52.54
7    4  treatment      56.48
8    5    control      48.72
9    5  treatment      51.25


## Query 5: Click position distribution

In [10]:
query5 = """
SELECT 
    "group",
    click_position,
    COUNT(*) as count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY "group"), 2) as pct
FROM search_events
WHERE click_position IS NOT NULL
GROUP BY "group", click_position
ORDER BY "group", click_position
"""

position_summary = pd.read_sql(query5, conn)
print("=== Click Position Distribution ===")
print(position_summary)
position_summary.to_csv('result_positions.csv', index=False)

=== Click Position Distribution ===
       group  click_position  count    pct
0    control             1.0   1980  40.51
1    control             2.0   1219  24.94
2    control             3.0    746  15.26
3    control             4.0    548  11.21
4    control             5.0    395   8.08
5  treatment             1.0   3002  55.07
6  treatment             2.0   1104  20.25
7  treatment             3.0    631  11.58
8  treatment             4.0    434   7.96
9  treatment             5.0    280   5.14
